# പാഠം 10 - ഉത്പാദനത്തിലെ AI ഏജന്റുകൾ

ഈ പാഠത്തിൽ നിങ്ങൾക്ക് Microsoft Agent Framework (Python) ഉപയോഗിച്ച് AI ഏജന്റുകൾക്കുള്ള **ഉത്പാദന മാതൃകകൾ** പഠിക്കാം. നാം ഉൾക്കൊള്ളുന്നവ:

- **Observability** — ഏജന്റ് ഇടപെടലുകളിൽ സമയമാപനവും ലോഗിംഗും ചേർക്കൽ
- **Evaluation** — പ്രതികരണ ഗുണമേന്മയ്ക്ക് സ്കോർ നൽകാൻ ഒരു ഇവാല്യുവേറ്റർ ഏജന്റ് ഉപയോഗിക്കൽ
- **Cost management** — ടോക്കൺ മെച്ചപ്പെടുത്തലിനുള്ളതും മോഡൽ തിരഞ്ഞെടുപ്പിനുള്ളതുമായ തന്ത്രങ്ങൾ

സന്നിവേശം ഒരു **യാത്രാ ഏജന്റ്** ആണ്, ഇത് ഉപയോക്താക്കളെ യാത്രാ പദ്ധതികൾ തയ്യാറാക്കാൻ സഹായിക്കുന്നു, അതിന്റെ മീതെ നിരീക്ഷണവും മൂല്യനിർണയവും ഏർപ്പെടുത്തിയിട്ടുണ്ട്.


## സജ്ജീകരണം


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ഉത്പാദന പരിഗണനകൾ

എഐ ഏജന്റുകളെ പ്രോട്ടോടൈപ്പുകളിൽ നിന്ന് പ്രൊഡക്ഷനിലേക്ക് മാറ്റുമ്പോൾ മൂന്ന് പ്രധാന തൂണുകൾക്കുള്ള സൂക്ഷ്മ ശ്രദ്ധ ആവശ്യമാണ്:

1. **നിരീക്ഷണക്ഷമത** — ഏജന്റ് എന്താണ് ചെയ്യുന്നത്, അത് എത്ര സമയം എടുക്കുന്നു, ഏത് ടൂളുകൾ അത് വിളിക്കുന്നു എന്നതിനെക്കുറിച്ച് ദൃശ്യത ഉണ്ടായിരിക്കണം. ട്രേസിംഗ്‌യും ലോഗിംഗും ഇല്ലാതെ പ്രൊഡക്ഷൻ പ്രശ്നങ്ങൾ ഡീബഗ് ചെയ്യുന്നത് ഏകദേശം അസാധ്യമാണ്.

2. **വിലയിരുത്തൽ** — ഓട്ടോമേറ്റഡ് ഗുണനിലവാര പരിശോധനകൾ കാലക്രമത്തിൽ ഏജന്റിന്റെ പ്രതികരണങ്ങൾ ശരിയായതും പൂർണ്ണവുമായതും സഹായകരവുമായതുമാണെന്ന് ഉറപ്പുനൽകുന്നു. ഒരു മൂല്യനിർണയ ഏജന്റ് നിർവചിച്ച മാനദണ്ഡങ്ങളുടെ അടിസ്ഥാനത്തിൽ പ്രതികരണങ്ങൾക്ക് സ്കോർ നൽകാൻ കഴിയും.

3. **ചെലവ് നിയന്ത്രണം** — ടോക്കൺ ഉപയോഗം നേരിട്ട് ചെലവിനെ ബാധിക്കുന്നു. പ്രോംപ്റ്റ് ഒപ്റ്റിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പ്, ക്യാഷിംഗ് പോലുള്ള തന്ത്രങ്ങൾ ഗുണമേന്മയെ ത്യജിക്കാതെ ചെലവുകൾ നിയന്ത്രിക്കാൻ സഹായിക്കുന്നു.


## ഒരു നിരീക്ഷണയോഗ്യമായ ഏജന്റ് നിർമ്മാണം

ഞങ്ങൾ യാത്രാ ഉപകരണങ്ങൾ നിർവചിക്കുകയും, ലേറ്റൻസി നിരീക്ഷിക്കാൻ ഏജന്റിന്റെ കോളിനെ ടൈമിംഗോടെയും ചുറ്റിപ്പൂട്ടുകയും ചെയ്യുന്നു. പ്രൊഡക്ഷനിൽ നിങ്ങൾ OpenTelemetry അല്ലെങ്കിൽ സമാനമായ ഒരു ട്രേസിംഗ് ബാക്ക്‌എൻഡുമായി സംയോജിപ്പിക്കും.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## അവലോകന മാതൃകകൾ

സാധാരണ ഉത്പാദന മാതൃകയിലൊരു രണ്ടാം ഏജന്റിനെ **മൂല്യനിർണയകൻ** ആയി ഉപയോഗിക്കുകയാണ്. മൂല്യനിർണയകൻ മുന്‍കൂട്ടി നിർവ്വചിച്ച മാനദണ്ഡങ്ങളായ പൂർണത, കൃത്യത, സഹായകത എന്നിവയുടെ അടിസ്ഥാനത്തിൽ പ്രാഥമിക ഏജന്റിന്റെ പ്രതികരണം സ്കോർ ചെയ്യുന്നു.

ഇതുവഴി സാധ്യമാകുന്നത്:
- ഉത്തരങ്ങൾ ഉപയോക്താക്കളിലേക്ക് എത്തുന്നതിന് മുമ്പ് സ്വയം പ്രവർത്തിക്കുന്ന ഗുണനിലവാര പരിശോധനകൾ
- പ്രോംപ്റ്റുകളും മോഡലുകളും മാറ്റപ്പെട്ടപ്പോൾ റഗ്രഷൻ കണ്ടെത്തൽ
- കാലക്രമത്തിൽ ഏജന്റിന്റെ പ്രകടനം തുടർച്ചയായി നിരീക്ഷണം


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ചെലവ് നിയന്ത്രണ തന്ത്രങ്ങൾ

പ്രൊഡക്ഷനിലെ AI ഏജന്റുകൾക്ക് ചിലവുകൾ നിയന്ത്രിക്കുന്നത് നിർണ്ണായകമാണ്. ഇവിടെ പ്രധാന തന്ത്രങ്ങൾ ഇവയാണ്:

| Strategy | Description |
|---|---|
| **പ്രോംപ്റ്റ് മെച്ചപ്പെടുത്തൽ** | സിസ്റ്റം നിർദ്ദേശങ്ങൾ സംക്ഷിപ്തമാക്കുക. ഇൻപുട്ട് ടോക്കണുകൾ കുറക്കാൻ ആവശ്യമില്ലാത്ത പശ്ചാത്തലം നീക്കം ചെയ്യുക. |
| **മോഡൽ തിരഞ്ഞെടുപ്പ്** | വർഗീകരണം അല്ലെങ്കിൽ എക്സ്ട്രാക്ഷൻ പോലുള്ള ലളിതമായ ജോലികൾക്കായി ചെറുതും വിലകുറഞ്ഞതുമായ മോഡലുകൾ ഉപയോഗിക്കുക (ഉദാ. GPT-4o-mini), സങ്കീർണമായ ചിന്തനത്തിനായി വലിയ മോഡലുകൾ സംരക്ഷിക്കുക. |
| **കാഷിംഗ്** | ടൂൾ ഫലങ്ങളും ആവർത്തിക്കുന്ന ക്വെറിയുകളും കാഷ് ചെയ്യുക, ആവർത്തിക്കുന്ന API കോൾകൾ ഒഴിവാക്കാൻ. |
| **ടോക്കൺ ബജറ്റുകൾ** | അപ്രതീക്ഷിതമായി ദൈര്‍ഘ്യമുള്ള പ്രതികരണങ്ങൾ തടയാൻ `max_tokens` പരിധികൾ സജ്ജമാക്കുക. |
| **ബാച്ചിംഗ്** | സാധ്യമെങ്കിൽ ഒരൊറ്റ API കോളിലേക്ക് പല ഉപയോക്തൃ ചോദ്യങ്ങളും കൂട്ടിച്ചേർക്കുക. |

വ്യവഹാരത്തിൽ, ഒരു ഘട്ടപദ്ധതി നല്ലതായാണ് പ്രവർത്തിക്കുന്നത്: എളുപ്പമുള്ള അഭ്യർത്തനങ്ങൾ വേഗതയുള്ള, ചെലവുകുറഞ്ഞ മോഡലിലേക്ക് അയച്ച് മാത്രം സങ്കീർണമായ ക്വെറികൾ കൂടുതൽ ശേഷിയുള്ള മോഡലിലേക്ക് ഉയർത്തുക.


## Summary

ഈ പാഠത്തിൽ നിങ്ങൾ എന്തെല്ലാം പഠിച്ചു എന്ന് കാണാം:

1. **നിരീക്ഷണ ശേഷി ചേർക്കുക** ഏജന്റ് ഇടപെടലുകളിൽ ടൈമിംഗും ലോഗിംഗും ഉപയോഗിച്ച് ട്രേസിംഗിനും മോണിറ്ററിംഗിനും വായ്പ്പുനിൽക്കുന്ന ഒരു അടിസ്ഥാനമൊരുക്കുക.
2. **എജന്റ് പ്രതികരണങ്ങളെ വിലയിരുത്തുക** പൂര്‍ണ്ണതക്ക്, കൃത്യതക്ക്, സഹായപരതയ്ക്ക് ആണ് സ്കോർ നൽകുന്നത് എന്ന രീതിയില്‍ ഒരു മൂല്യനിർണയ എജന്റ് ഉപയോഗിച്ച് സ്വയംമാറ്റമായി നിർവഹിക്കുക.
3. **ചെലവുകൾ നിയന്ത്രിക്കുക** പ്രോംപ്റ്റ് ഓപ്റ്റിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പ്, കാഷിംഗ്, ടോക്കൺ ബഡ്ജറ്റുകൾ എന്നിവയിലൂടെ.

ഈ ഉൽപ്പാദന മാതൃകകൾ നിങ്ങളുടെ AI എജന്റുകൾ വിശ്വസനീയവും, അളക്കാവുന്നതുമായും വ്യാപകമായി ചെലവ്-ഫലപ്രദവുമായും ഉണ്ടാകാൻ സഹായിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
അറിയിപ്പ്:
ഈ രേഖ AI വിവർത്തന സേവനം Co-op Translator (https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് വിവർത്തനം ചെയ്തതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്ക് ശ്രമിക്കുമ്പോഴും, യാന്ത്രിക വിവർത്തനങ്ങളിൽ പിശകുകൾ അല്ലെങ്കിൽ വീഴ്ചകൾ ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. മാതൃഭാഷയിലുള്ള മൂല രേഖയാണ് അധികാരപരമായ ഉറവിടമായി കണക്കാക്കപ്പെടേണ്ടത്. നിർണ്ണായകമായ വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മാനവ വിവർത്തനം ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിച്ചതിന് ശേഷം ഉണ്ടായ οποιαδήποτε തെറ്റിദ്ധാരണകൾക്കും അല്ലെങ്കിൽ വ്യാഖ്യാനപിശകുകൾക്കും ഞങ്ങൾ ഉത്തരവാദികൾ അല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
